# 02 - Build the Loop Yourself (OpenAI SDK)

## Scenario: Northstar Incident Response Assistant

Before using heavy multi-agent frameworks like LangGraph or CrewAI, it's critical to understand the foundation: **The Agent Loop**. At its core, an agent is just a `while` loop around an LLM and some tools. The model proposes tools to call, your application executes them and returns the results, and the model proposes the next step.

In this module, you will build a robust Agent Loop using the official `openai` Python SDK. We will go beyond a simple loop and implement:
1. Tool definitions and execution.
2. The Observe → Decide → Act loop.
3. **Safety Budgets**: Preventing infinite loops via step counts.
4. **Error Recovery**: Handling hallucinated tools or execution crashes gracefully without crashing the loop.
5. **Cost Tracking**: Monitoring token usage across a multi-step execution.

In [1]:
import json
import os
from openai import OpenAI

# Initialize the OpenAI client (Requires OPENAI_API_KEY environment variable)
# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Let's define some dummy systems for Northstar's incident response
def get_service_status(service: str) -> str:
    """Returns the current health status of a service."""
    print(f"  🔧 [Tool] Checking status for {service}...")
    if service.lower() == "checkout":
        return json.dumps({"status": "degraded", "latency_ms": 4500})
    return json.dumps({"status": "healthy", "latency_ms": 120})

def search_incidents(query: str) -> str:
    """Searches the incident database for ongoing issues."""
    print(f"  🔧 [Tool] Searching incidents for '{query}'...")
    return json.dumps([
        {"id": "INC-8801", "title": "EU region checkout latency spikes", "severity": "high"}
    ])

def restart_service(service: str) -> str:
    """Restarts a given service. Extremely dangerous in production."""
    print(f"  🚨 [Tool] Attempting to restart {service}...")
    # We purposefully make this tool fail to demonstrate error recovery later!
    raise RuntimeError("Permission Denied: Automated service restarts are disabled during peak hours.")

# Map the function names to their actual Python implementations
available_tools = {
    "get_service_status": get_service_status,
    "search_incidents": search_incidents,
    "restart_service": restart_service
}


⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Defining Tool Schemas

To let the LLM know these tools exist, we define them using JSON schema syntax.

In [2]:
# OpenAI Tool Schemas
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_service_status",
            "description": "Returns the current health status of a service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service": {"type": "string", "description": "The name of the service, e.g., 'checkout'"}
                },
                "required": ["service"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_incidents",
            "description": "Searches the incident database for ongoing issues.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Restarts a given service to mitigate severe latency or crashes.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service": {"type": "string", "description": "Service to restart"}
                },
                "required": ["service"]
            }
        }
    }
]

## 2. The Robust Agent Loop

A naive loop just calls the function and assumes it works. A **production loop** catches Python exceptions, formats them into a message, and sends them back to the LLM so the LLM knows the tool failed and can try something else!

In [3]:
def run_agent_loop(user_request: str, max_steps: int = 5):
    messages = [
        {"role": "system", "content": "You are a Northstar support DevOps agent. Use tools to investigate issues before recommending action or attempting mitigations."},
        {"role": "user", "content": user_request}
    ]
    
    total_tokens = 0
    print(f"--- Starting Agent Loop (Budget: {max_steps} steps) ---")
    
    for step in range(max_steps):
        print(f"\nStep {step + 1}: LLM is thinking...")
        
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                tools=tools,
                temperature=0
            )
            
            # Track cost (token usage)
            if response.usage:
                total_tokens += response.usage.total_tokens
                print(f"  💰 Usage so far: {total_tokens} tokens")
                
            message = response.choices[0].message
            messages.append(message)
            
            # Stop condition 1: The model replied with a final text answer
            if not message.tool_calls:
                print("\n✅ [Agent Final Answer]:", message.content)
                return messages
            
            # Execute requested tools
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                
                try:
                    args = json.loads(tool_call.function.arguments)
                    
                    if func_name not in available_tools:
                        raise ValueError(f"Tool '{func_name}' does not exist.")
                        
                    # Application Code executes the tool
                    func_to_call = available_tools[func_name]
                    result = func_to_call(**args)
                    
                except Exception as e:
                    # CRITICAL: We don't crash the Python script! 
                    # We catch the error and return it to the LLM.
                    print(f"  ⚠️ [Tool Execution Failed] {e}")
                    result = f"Error executing tool: {str(e)}"
                
                # Append observation (or error) back to state
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": func_name,
                    "content": result
                })
                
        except Exception as e:
            print(f"API Error (Did you set OPENAI_API_KEY?): {e}")
            return messages
            
    # Stop condition 2: Loop budget exhausted
    print("\n🛑 [Agent Terminated]: Maximum steps reached. Loop budget exhausted.")
    return messages


## 3. Running the Loop: Success Path

Let's test the agent loop with a typical request. It should check the status and search incidents before replying.

In [4]:
_ = run_agent_loop("Customers are reporting checkout failures. Check the status and search incidents.")

--- Starting Agent Loop (Budget: 5 steps) ---

Step 1: LLM is thinking...
API Error (Did you set OPENAI_API_KEY?): 'MockCompletion' object has no attribute 'usage'


## 4. Running the Loop: Error Recovery

What happens if the LLM tries to do something dangerous and the tool throws an exception? Because we built a robust loop, the LLM will see the `RuntimeError`, realize it can't restart the server, and apologize instead of crashing.

In [5]:
# Watch the agent try to restart the service, fail, and self-correct!
_ = run_agent_loop("Checkout is failing. Please just restart the checkout service immediately to fix it.")

--- Starting Agent Loop (Budget: 5 steps) ---

Step 1: LLM is thinking...
API Error (Did you set OPENAI_API_KEY?): 'MockCompletion' object has no attribute 'usage'


## 5. Runaway Loops and Step Budgets

If we tell the model to do an impossible task, it might get confused and loop forever. We simulate this by giving a stubborn instruction, but forcing `max_steps=2`. Notice how the loop halts safely before burning through too many API tokens.

In [6]:
_ = run_agent_loop("Keep checking the checkout status continuously until it is completely healthy.", max_steps=2)

--- Starting Agent Loop (Budget: 2 steps) ---

Step 1: LLM is thinking...
API Error (Did you set OPENAI_API_KEY?): 'MockCompletion' object has no attribute 'usage'


## Watch For

- **Crashing on Tool Errors**: Never let a `KeyError` or `RuntimeError` in your tool functions crash the `while` loop. Catch it and send it to the LLM. The LLM is surprisingly good at reading error messages and fixing its JSON arguments.
- **Token Inflation**: Because we append every tool result to `messages`, the context window grows every step. The token cost of Step 4 is much higher than Step 1.
- **Budget Exhaustion**: Always have a hard `max_steps` to prevent a confused model from draining your API credits.

## Checkpoint

**1. In a raw Agent Loop, what is the best practice if a tool function raises a Python Exception?**
- A) Crash the program immediately so the developer knows.
- B) Catch the exception, format it as a string, and append it as a `tool` observation so the LLM can see the error.
- C) Silently ignore it and continue the loop.
- D) Restart the OpenAI client.

**2. Why do the tokens consumed per step increase as the loop progresses?**
- A) OpenAI charges more for later steps.
- B) The LLM gets slower over time.
- C) The `messages` array contains the entire history of the conversation, so the LLM has to read a longer prompt on every iteration.
- D) Tools use up tokens when they execute locally.
